![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Custom Indicator Warm-Up Research

Define a custom indicator and warm it from typed daily TradeBar history.

### Set Up QuantBook

Create a daily SPY subscription for the custom indicator updates.

In [ ]:
qb = QuantBook()
qb.set_start_date(2024, 9, 1)
qb.settings.seed_initial_prices = True
equity = qb.add_equity("SPY", Resolution.DAILY)

### Define Indicator

Implement annualized volatility by chaining LogReturn into StandardDeviation.

In [ ]:
class CustomVolatility(PythonIndicator):

    def __init__(self, period, trading_days_per_year=252):
        super().__init__()
        self.warm_up_period = period
        self._trading_days_per_year = trading_days_per_year
        self.value = 0
        self._log_return = LogReturn(1)
        self._standard_deviation = IndicatorExtensions.of(StandardDeviation(period), self._log_return)

    def update(self, input_: BaseData):
        # Annualized log-return volatility.
        price = input_.value
        if price <= 0:
            return
        self._log_return.update(input_.end_time, price)
        if self.is_ready:
            self.value = self._standard_deviation.current.value * math.sqrt(self._trading_days_per_year) * 100.0
        return self.is_ready

    @property
    def is_ready(self) -> bool:
        return self._standard_deviation.is_ready

### Warm Custom Indicator

Attach the volatility indicator to the Equity, warm it, and register it for future updates.

In [ ]:
equity.volatility = CustomVolatility(3*21)
# Warm the custom indicator with typed daily trade bars.
history = qb.history[TradeBar](equity, equity.volatility.warm_up_period, Resolution.DAILY)
for bar in history:
    equity.volatility.update(bar)
qb.register_indicator(equity, equity.volatility)

### Verify Readiness

Confirm the custom indicator is ready and has a positive volatility value.

In [ ]:
print(f"Volatility ready: {equity.volatility.is_ready}")
print(f"Volatility value: {equity.volatility.value:.2f}")
print(f"Previous value: {equity.volatility.previous.value:.2f}")
assert equity.volatility.is_ready
assert equity.volatility.value > 0

### Build Time Series

Replay a longer history window and store raw volatility values.

In [ ]:
series_volatility = CustomVolatility(3*21)
value_rows = []
signal_rows = []
for bar in qb.history[TradeBar](equity, 180, Resolution.DAILY):
    series_volatility.update(bar)
    if not series_volatility.is_ready:
        continue
    value_rows.append({"time": bar.end_time, "volatility": series_volatility.value})
    signal_rows.append({
        "time": bar.end_time,
        "signal": 1 if series_volatility.value > series_volatility.previous.value else 0
    })

indicator_values = pd.DataFrame(value_rows).set_index("time")
indicator_values.tail()

### Signal Series

Display the 1/0 signal generated when volatility is increasing.

In [ ]:
signals = pd.DataFrame(signal_rows).set_index("time")
signals.tail()